link zum Chat: https://chatgpt.com/c/69ea070a-4f10-832b-809a-c5d9cd3dbd69

In [21]:
import pandas as pd
import numpy as np

df = pd.read_csv("/Users/charlottewindlin/4.Semester/spotify_dataset.csv")

# clean
df = df.drop(columns=["Unnamed: 0"])
df = df.drop_duplicates(subset="track_id")

print("Shape:", df.shape)
df.head()

Shape: (89741, 20)


,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,1,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,0,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,0,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


In [22]:
# define slider features
SLIDER_FEATURES = [
    "danceability",
    "energy",
    "valence",
    "acousticness",
    "tempo"
]

# bins for 0–1 features
bins_01 = np.arange(0, 1.1, 0.1)

for col in ["danceability", "energy", "valence", "acousticness"]:
    df[col + "_bin"] = pd.cut(
        df[col],
        bins=bins_01,
        labels=False,
        include_lowest=True
    )

# bins for tempo (in BPM)
tempo_bins = np.arange(0, 250, 10)

df["tempo_bin"] = pd.cut(
    df["tempo"],
    bins=tempo_bins,
    labels=False,
    include_lowest=True
)

# check result
df[[
    "danceability", "danceability_bin",
    "energy", "energy_bin",
    "tempo", "tempo_bin"
]].head()

,danceability,danceability_bin,energy,energy_bin,tempo,tempo_bin
0,0.676,6,0.4610,4,87.917,8.0
1,0.420,4,0.1660,1,77.489,7.0
2,0.438,4,0.3590,3,76.332,7.0
3,0.266,2,0.0596,0,181.740,18.0
4,0.618,6,0.4430,4,119.949,11.0


In [23]:
lookup = df.groupby([
    "danceability_bin",
    "energy_bin",
    "valence_bin",
    "acousticness_bin",
    "tempo_bin"
]).agg(
    avg_popularity=("popularity", "mean"),
    count=("popularity", "size")
).reset_index()

print("Lookup table shape:", lookup.shape)
lookup.head()

Lookup table shape: (27889, 7)


,danceability_bin,energy_bin,valence_bin,acousticness_bin,tempo_bin,avg_popularity,count
0,0,0,0,0,0.0,37.923077,26
1,0,0,0,0,5.0,33.500000,2
2,0,0,0,0,7.0,36.250000,4
3,0,0,0,0,8.0,18.666667,3
4,0,0,0,0,17.0,0.000000,2


In [24]:
# check distribution of counts
lookup["count"].describe()

count    27889.000000
mean         3.217756
std          5.632800
min          1.000000
25%          1.000000
50%          2.000000
75%          3.000000
max        114.000000
Name: count, dtype: float64

In [25]:
lookup_filtered = lookup[lookup["count"] >= 20].copy()

print("Before:", lookup.shape)
print("After:", lookup_filtered.shape)

Before: (27889, 7)
After: (572, 7)


In [26]:
lookup_filtered["count"].describe()

count    572.000000
mean      34.277972
std       14.709255
min       20.000000
25%       23.000000
50%       30.000000
75%       41.000000
max      114.000000
Name: count, dtype: float64

In [12]:
def bin_to_value(bin_index, step):
    return bin_index * step + step / 2

In [27]:
lookup_filtered["danceability_val"] = lookup_filtered["danceability_bin"].apply(lambda x: bin_to_value(x, 0.1))
lookup_filtered["energy_val"]       = lookup_filtered["energy_bin"].apply(lambda x: bin_to_value(x, 0.1))
lookup_filtered["valence_val"]      = lookup_filtered["valence_bin"].apply(lambda x: bin_to_value(x, 0.1))
lookup_filtered["acousticness_val"] = lookup_filtered["acousticness_bin"].apply(lambda x: bin_to_value(x, 0.1))
lookup_filtered["tempo_val"]        = lookup_filtered["tempo_bin"].apply(lambda x: bin_to_value(x, 10))

In [28]:
lookup_filtered[[
    "danceability_bin", "danceability_val",
    "tempo_bin", "tempo_val"
]].head()

,danceability_bin,danceability_val,tempo_bin,tempo_val
0,0,0.05,0.0,5.0
5,0,0.05,0.0,5.0
34,0,0.05,0.0,5.0
36,0,0.05,6.0,65.0
37,0,0.05,7.0,75.0


In [29]:
lookup_filtered.to_csv("/Users/charlottewindlin/4.Semester/PODSV/ad24-1-fancyproject/edab", index=False)

print("Slider lookup table saved!")

Slider lookup table saved!
